```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11 done;
    class A12 current;
```

# Notebook 12 — Semantic Shift: Tracking Conceptual Change with Embeddings

This notebook develops a **mini-study of semantic shift** using the chunk embeddings generated earlier in the course. The main question is not whether we can *prove* that a concept changed meaning, but whether we can identify **structured changes in semantic neighborhoods over time** and discuss them responsibly.

We use a deliberately cautious workflow:
- retrieve **embedding neighborhoods** for target concepts within each `time_bin`
- compare those neighborhoods across periods
- inspect representative chunks rather than trusting metrics alone
- run lightweight robustness checks
- distinguish **semantic change** from other explanations such as corpus composition, genre shifts, or uneven bin sizes

This notebook is exploratory and interpretive. It should support claims such as:

> “The neighborhood around *virtue* looks different in later periods, and this pattern is stable across multiple query settings.”

It should **not** support stronger claims such as:

> “The meaning of *virtue* objectively changed in year X.”

## Learning goals

By the end of this notebook, students should be able to:

- explain what an embedding-based semantic shift analysis can and cannot show
- define a target concept and retrieve its nearest semantic neighbors across time bins
- compare neighborhood centroids across periods
- identify representative passages that support or challenge a shift claim
- discuss confounds such as corpus composition, frequency imbalance, and query wording
- perform at least one robustness check before reporting a semantic-shift result

## Method note: what counts as “semantic shift” here?

In this notebook, we operationalize semantic shift as **change in the embedding neighborhood of a concept query across time bins**. This is a practical and CPU-friendly approximation, but it comes with important caveats:

- we are comparing **retrieved neighborhoods**, not directly aligning historical word senses
- differences may reflect **changing corpora**, not only changing meanings
- a query such as `reason` may retrieve different subtopics in different periods
- broad philosophical corpora often mix **genre, translation style, authorial voice, and conceptual history**

For that reason, every quantitative result in this notebook should be paired with:
1. a **representative chunk inspection**
2. a **bin-size check**
3. at least one **robustness comparison**

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import json
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt
import seaborn as sns

# Optional interactive plots
try:
    import plotly.express as px
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

# Sentence-transformers for encoding concept queries
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except Exception:
    HAS_ST = False
    print('sentence-transformers not available.')
    print('Install with: pip install sentence-transformers')

In [ ]:
# ------------------------------------------------------------
# Paths and configuration
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'
CACHE_DIR = PROJECT_ROOT / 'cache'

for p in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Notebook 08 artifacts
CHUNKS_PATH = TABLES_DIR / 'nb08-chunks.parquet'
EMB_PATH = CACHE_DIR / 'nb08-chunk_embeddings.npy'

# Document index for metadata checks
DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'

# Query / neighborhood settings
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
TOP_K_NEIGHBORS = 15

print('CHUNKS_PATH:', CHUNKS_PATH)
print('EMB_PATH:', EMB_PATH)

## Load cached chunks and embeddings from Notebook 08

We reuse the paragraph-sized chunks and cached embedding matrix produced earlier. This is essential for a CPU-friendly workflow: the expensive embedding step should not be repeated in every notebook.

If these files are missing, run Notebook 08 first.

In [ ]:
if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f'Missing chunks table: {CHUNKS_PATH}')

if not EMB_PATH.exists():
    raise FileNotFoundError(f'Missing embedding matrix: {EMB_PATH}')

chunks_df = pd.read_parquet(CHUNKS_PATH)
E = np.load(EMB_PATH)

if len(chunks_df) != len(E):
    raise ValueError(f"Mismatch: chunks_df has {len(chunks_df)} rows but embeddings have {len(E)} vectors")

print('Chunks:', f'{len(chunks_df):,}')
print('Embedding matrix shape:', E.shape, '\n')
display(chunks_df.head())

In [ ]:
# If there is a mismatch between chunk_df and the embedding
# uncomment and run the following code

# # Load the chunks index used for the embedding
# chunks_df = pd.read_parquet(Path('./analysis/tables/nb08-chunk_emb_index.parquet'))  # or .csv with index=True

# # Load the embeddings DataFrame (which has the full index)
# embedding_df = pd.read_parquet(Path('./analysis/tables/nb08-emb_index.parquet'))

# # Extract the embeddings
# E = embedding_df.loc[chunks_df.index].values

# print(f"\nChunks: {len(chunks_df):,}")
# print(f"Embedding matrix shape: {E.shape}\n")
# display(chunks_df.head())

## Inspect time-bin coverage

Before making any claim about semantic shift, we should know how much material exists in each period. Sparse bins can produce unstable neighborhoods and misleading comparisons.

In [ ]:
def _bin_sort_key(x) -> float:
    if hasattr(x, 'left'):
        return float(x.left)
    s = str(x).replace('–', '-').replace('−', '-')
    m = re.search(r'-?\d+(?:\.\d+)?', s)
    return float(m.group(0)) if m else float('inf')

bin_counts = (
    chunks_df.dropna(subset=['time_bin'])
             .groupby('time_bin')
             .size()
             .rename('n_chunks')
             .reset_index()
)

bin_counts = bin_counts.sort_values('time_bin', key=lambda s: s.map(_bin_sort_key))
display(bin_counts)

plt.figure(figsize=(10, 4))
sns.barplot(data=bin_counts, x='time_bin', y='n_chunks', color='teal')
plt.xticks(rotation=45, ha='right')
plt.title('Chunks per time bin')
plt.xlabel('Time bin')
plt.ylabel('Number of chunks')
plt.tight_layout()
plt.show()

## Load the embedding model for query encoding

The chunk embeddings are already cached. We only need the model here to encode short **concept queries** such as `reason` or `virtue` into the same embedding space.

### Reflection question
Why is encoding five concept queries much cheaper than re-embedding the entire corpus?

In [ ]:
if not HAS_ST:
    raise RuntimeError("sentence-transformers is required to encode concept queries in this notebook.")

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print('Loaded model:', EMBEDDING_MODEL_NAME)

## Helper functions

We define a small set of reusable helpers for:
- encoding concept queries
- retrieving top neighbors within a time bin
- computing neighborhood centroids
- measuring consecutive-bin shift
- extracting representative chunks
- summarizing neighborhood vocabulary with lightweight TF–IDF

In [ ]:
def encode_query(query: str) -> np.ndarray:
    """Encode a short query string into the same embedding space as the chunk embeddings."""
    q = model.encode([query], normalize_embeddings=True)
    return np.asarray(q[0])


def get_bin_mask(chunks: pd.DataFrame, bin_label) -> np.ndarray:
    """Return a boolean mask for one time bin."""
    return chunks['time_bin'].astype(str).to_numpy() == str(bin_label)


def top_neighbors_in_bin(
    query_vec: np.ndarray,
    chunks: pd.DataFrame,
    E: np.ndarray,
    bin_label,
    k: int = 15,
) -> pd.DataFrame:
    """Retrieve the top-k most similar chunks to a query within a given time bin."""
    mask = get_bin_mask(chunks, bin_label)
    idx = np.where(mask)[0]

    if len(idx) == 0:
        return pd.DataFrame()

    sims = cosine_similarity(query_vec.reshape(1, -1), E[idx]).ravel()
    order = np.argsort(-sims)[: min(k, len(idx))]

    out = chunks.iloc[idx[order]].copy()
    out['cosine_sim'] = sims[order]
    return out


def centroid_from_neighbors(
    query_vec: np.ndarray,
    chunks: pd.DataFrame,
    E: np.ndarray,
    bin_label,
    k: int = 15,
) -> np.ndarray | None:
    """Build a centroid from the top-k retrieved neighbors within a bin."""
    mask = get_bin_mask(chunks, bin_label)
    idx = np.where(mask)[0]

    if len(idx) == 0:
        return None

    sims = cosine_similarity(query_vec.reshape(1, -1), E[idx]).ravel()
    order = np.argsort(-sims)[: min(k, len(idx))]
    neigh = E[idx[order]]

    c = neigh.mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-12)
    return c


def concept_shift_over_time(
    concept: str,
    chunks: pd.DataFrame,
    E: np.ndarray,
    labels: list,
    k: int = 15,
) -> pd.DataFrame:
    """Compute consecutive-bin semantic shift for one concept using neighborhood centroids."""
    q = encode_query(concept)

    centroids = []
    valid_labels = []

    for b in labels:
        c = centroid_from_neighbors(q, chunks, E, b, k=k)
        if c is not None:
            centroids.append(c)
            valid_labels.append(str(b))

    if len(centroids) < 2:
        return pd.DataFrame(columns=['concept', 'time_bin', 'prev_bin', 'cosine_sim_to_prev', 'cosine_dist_to_prev'])

    C = np.vstack(centroids)
    sims = np.sum(C[1:] * C[:-1], axis=1)
    dists = 1 - sims

    rows = []
    for i in range(1, len(valid_labels)):
        rows.append({
            'concept': concept,
            'prev_bin': valid_labels[i-1],
            'time_bin': valid_labels[i],
            'cosine_sim_to_prev': float(sims[i-1]),
            'cosine_dist_to_prev': float(dists[i-1]),
        })

    return pd.DataFrame(rows)


def top_terms_from_texts(texts: list[str], k: int = 15) -> pd.DataFrame:
    """Summarize a small neighborhood with TF–IDF top terms for interpretive inspection."""
    if len(texts) == 0:
        return pd.DataFrame(columns=['term', 'mean_tfidf'])

    vec = TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        min_df=1,
        max_features=3000,
        ngram_range=(1, 2),
    )
    X = vec.fit_transform(texts)
    mean_vec = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vec.get_feature_names_out())
    top = np.argsort(-mean_vec)[: min(k, len(terms))]
    return pd.DataFrame({'term': terms[top], 'mean_tfidf': mean_vec[top]})


def jaccard_overlap(a: list, b: list) -> float:
    """Jaccard overlap between two collections of identifiers."""
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / len(sa | sb)

## Choose usable time bins

For neighborhood comparison, we exclude bins with too few chunks. This is not a perfect solution, but it reduces instability.

### Reflection question
What happens to a semantic-shift claim if one of the compared periods contains very little text?

In [ ]:
MIN_CHUNKS_PER_BIN = 1

usable_bins = (
    bin_counts.loc[bin_counts['n_chunks'] >= MIN_CHUNKS_PER_BIN, 'time_bin']
              .tolist()
)

usable_bins = sorted(usable_bins, key=_bin_sort_key)

print('Usable bins:', usable_bins)

## Retrieve semantic neighborhoods for selected concepts

For each concept, we retrieve the top neighbors within each time bin. This lets us inspect:
- which passages are most similar to the query
- whether those passages differ substantially across periods
- whether neighborhood changes are interpretable or merely noisy

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
TARGET_CONCEPTS = [
# Add your selected target concepts
]

In [ ]:
neighbor_rows = []

for concept in TARGET_CONCEPTS:
    q = encode_query(concept)
    for b in usable_bins:
        neigh = top_neighbors_in_bin(q, chunks_df, E, b, k=TOP_K_NEIGHBORS)
        for row in neigh.itertuples(index=False):
            neighbor_rows.append({
                'concept': concept,
                'time_bin': str(b),
                'pg_id': getattr(row, 'pg_id', pd.NA),
                'title': getattr(row, 'title', pd.NA),
                'chunk_index': getattr(row, 'chunk_index', pd.NA),
                'cosine_sim': getattr(row, 'cosine_sim', np.nan),
                'text': getattr(row, 'text', ''),
            })

neighbors = pd.DataFrame(neighbor_rows)
print('Neighborhood rows:', f'{len(neighbors):,}')
display(neighbors.head(20))

## Inspect representative chunks

Quantitative similarities are not enough. We should read actual chunks to see whether a concept query retrieves something meaningful in each period.

Try changing the `CONCEPT_TO_INSPECT` and `BIN_TO_INSPECT` values below.

In [ ]:
CONCEPT_TO_INSPECT = TARGET_CONCEPTS[0]
BIN_TO_INSPECT = usable_bins[0]

example_neigh = (
    neighbors.query("concept == @CONCEPT_TO_INSPECT and time_bin == @BIN_TO_INSPECT")
             .sort_values('cosine_sim', ascending=False)
             .head(8)
)

print('Concept:', CONCEPT_TO_INSPECT)
print('Time bin:', BIN_TO_INSPECT)
display(example_neigh[['cosine_sim', 'pg_id', 'title', 'chunk_index', 'text']])

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Explore different concepts and bins using the code above

## Consecutive-bin semantic shift scores

We now summarize change over time by:
1. retrieving the top-k neighbors for a concept in each bin
2. computing a neighborhood centroid
3. measuring cosine distance to the **previous** bin

Higher cosine distance means greater change between adjacent periods.

In [ ]:
shift_frames = []

for concept in TARGET_CONCEPTS:
    shift_df = concept_shift_over_time(
        concept=concept,
        chunks=chunks_df,
        E=E,
        labels=usable_bins,
        k=TOP_K_NEIGHBORS,
    )
    shift_frames.append(shift_df)

shift = pd.concat(shift_frames, ignore_index=True)
display(shift.head(20))

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=shift,
    x='time_bin',
    y='cosine_dist_to_prev',
    hue='concept',
    marker='o'
)
plt.xticks(rotation=45, ha='right')
plt.title('Consecutive-bin semantic shift by concept')
plt.xlabel('Time bin')
plt.ylabel('Cosine distance to previous bin')
plt.tight_layout()
plt.show()

## Neighborhood drift via overlap

We can also compare neighborhood or neighboring vectors to complement the vector-to-vector comparison.

### The idea

For a given query (e.g. a concept like "reason"), we can retrieve its **top-*k* nearest chunks** — the *k* passages of text whose embeddings are most similar to it — separately within each time-bin. If the concept's meaning is stable across periods, we'd expect it to keep "the same kind of company": similar passages should keep showing up among its nearest neighbors from one bin to the next. If the concept's meaning is drifting, the passages it neighbors should change, even if we haven't measured the vector shift directly.

We quantify "how much the neighborhood changed" using **Jaccard overlap**, a standard measure of similarity between two sets:

Jaccard(A, B) = |A ∩ B| / |A ∪ B|

— the number of chunk IDs the two top-*k* lists have in common, divided by the total number of distinct chunk IDs across both lists. This gives a score between 0 and 1:

- **1** → the two bins retrieve exactly the same top chunks — a completely stable neighborhood.
- **0** → the two bins retrieve entirely different chunks — no overlap at all.

Comparing this score bin-to-bin across the corpus's timespan gives us a simple drift signal: a sharp drop in overlap between two adjacent periods suggests something changed in what the concept is "near."

### Why this is only a proxy, not proof of semantic drift

A drop in overlap tells us the *identity* of nearby passages changed — but that's a symptom with several possible causes, and semantic change is only one of them:

- **Semantic drift** — the concept's meaning genuinely shifted, so it's now closer to different content. This is the signal we're actually hoping to detect.
- **Topic turnover** — the *subject matter* being written about shifted (e.g. more texts on ethics, fewer on metaphysics), which can move a concept's neighbors around even if its meaning didn't change.
- **Author turnover** — a different set of authors becomes prominent in the later bin, bringing distinct vocabulary and style that shifts which passages rank as "nearest," independent of meaning.
- **Data sparsity** — if a bin contains few documents, the top-*k* neighbors are estimated from a small, noisy sample, and low overlap may just reflect unreliable estimates rather than a real pattern.

### How to use this measure responsibly

Because of these confounds, a low overlap score should be treated as a **prompt for further investigation**, not a conclusion. Look at the actual retrieved chunks before and after a drop: does the *content* look meaningfully different, or does it look like the same idea expressed by different authors, or a change in how much material happens to exist in that bin? Cross-checking overlap-based drift signals against the embedding-based drift measures from earlier in the notebook — do they agree on *where* drift occurs? — is a good way to build more confidence in any pattern you report.

In [ ]:
overlap_rows = []

for concept in TARGET_CONCEPTS:
    q = encode_query(concept)
    prev_ids = None
    prev_bin = None

    for b in usable_bins:
        neigh = top_neighbors_in_bin(q, chunks_df, E, b, k=TOP_K_NEIGHBORS)
        ids = neigh['chunk_index'].tolist()

        if prev_ids is not None:
            overlap_rows.append({
                'concept': concept,
                'prev_bin': str(prev_bin),
                'time_bin': str(b),
                'jaccard_overlap': jaccard_overlap(prev_ids, ids),
            })

        prev_ids = ids
        prev_bin = b

overlap = pd.DataFrame(overlap_rows)
display(overlap.head(20))

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=overlap,
    x='time_bin',
    y='jaccard_overlap',
    hue='concept',
    marker='o'
)
plt.xticks(rotation=45, ha='right')
plt.title('Neighborhood overlap across consecutive bins')
plt.xlabel('Time bin')
plt.ylabel('Jaccard overlap of top retrieved chunks')
plt.tight_layout()
plt.show()

## Interpretive summaries: top TF–IDF terms inside a neighborhood

Embedding neighborhoods are not directly interpretable at the dimension level. One useful bridge is to summarize the retrieved neighborhood texts with a lightweight TF–IDF analysis.

This does **not** explain the embedding mechanically. Instead, it helps us describe what the retrieved passages are about.

In [ ]:
CONCEPT_TO_SUMMARIZE = TARGET_CONCEPTS[0]

for b in usable_bins[: min(4, len(usable_bins))]:
    texts = (
        neighbors.query("concept == @CONCEPT_TO_SUMMARIZE and time_bin == @b")
                 .sort_values('cosine_sim', ascending=False)['text']
                 .tolist()
    )
    print(f"\nConcept: {CONCEPT_TO_SUMMARIZE} | Bin: {b}")
    display(top_terms_from_texts(texts, k=12))

## Robustness check 1: sensitivity to neighborhood size

If the story changes dramatically when `k` changes from 10 to 20, then the result is not very stable. Here we compare shift curves under multiple neighborhood sizes.

In [ ]:
K_VALUES = [5, 10, 15, 25]

robust_rows = []

for concept in TARGET_CONCEPTS:
    for k in K_VALUES:
        tmp = concept_shift_over_time(
            concept=concept,
            chunks=chunks_df,
            E=E,
            labels=usable_bins,
            k=k,
        )
        tmp['k'] = k
        robust_rows.append(tmp)

robust_shift = pd.concat(robust_rows, ignore_index=True)
display(robust_shift.head(20))

In [ ]:
g = sns.FacetGrid(robust_shift, col='concept', col_wrap=2, height=3.5, sharey=False)
g.map_dataframe(sns.lineplot, x='time_bin', y='cosine_dist_to_prev', hue='k', marker='o')
g.add_legend()
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=45)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Robustness check: semantic shift under multiple neighborhood sizes')
plt.show()

## Robustness check 2: query wording

Embedding-based studies can be sensitive to the exact query string. For example, `freedom` and `liberty` may retrieve overlapping but non-identical neighborhoods.

Below you can define alternative queries for a concept and compare the resulting shift curves.

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
QUERY_VARIANTS = {
    'freedom': ['freedom', 'liberty'],
    'reason': ['reason', 'practical reason'],
    # Add query variants
}

variant_rows = []

for concept, variants in QUERY_VARIANTS.items():
    for qtext in variants:
        tmp = concept_shift_over_time(
            concept=qtext,
            chunks=chunks_df,
            E=E,
            labels=usable_bins,
            k=TOP_K_NEIGHBORS,
        )
        tmp['base_concept'] = concept
        tmp['query_variant'] = qtext
        variant_rows.append(tmp)

variant_shift = pd.concat(variant_rows, ignore_index=True)
display(variant_shift.head(20))

In [ ]:
if len(variant_shift):
    g = sns.FacetGrid(variant_shift, col='base_concept', col_wrap=2, height=3.5, sharey=False)
    g.map_dataframe(
        sns.lineplot,
        x='time_bin',
        y='cosine_dist_to_prev',
        hue='query_variant',
        marker='o'
    )
    g.add_legend()
    for ax in g.axes.flatten():
        ax.tick_params(axis='x', rotation=45)
    g.fig.subplots_adjust(top=0.88)
    g.fig.suptitle('Robustness check: semantic shift under alternative query wording')
    plt.show()

## Optional interactive view

With Plotly we can inspect the shift curves interactively.

In [ ]:
fig = px.line(
    shift,
    x='time_bin',
    y='cosine_dist_to_prev',
    color='concept',
    markers=True,
    title='Interactive semantic shift view'
)
fig.show(renderer="notebook")


## Bootstrap confidence intervals for time-bin comparisons

As in Notebook 03, we add bootstrap confidence intervals here as a resampling-based robustness check, so that differences in period-level embedding similarity are interpreted not only as point estimates but also in relation to their empirical uncertainty.

In [ ]:
# ------------------------------------------------------------
# Bootstrap confidence intervals for query-to-chunk similarity by time bin
# Resamples chunks within each time bin
# ------------------------------------------------------------
TARGET_QUERY = "virtue"
N_BOOT = 1000
RANDOM_STATE = 42


def bin_start(label) -> int:
    s = str(label)
    m = re.search(r"-?\d+", s.replace("–", "-"))
    return int(m.group(0)) if m else 10**9


if "model" not in globals():
    print("Loading embedding model for the query...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

q = model.encode([TARGET_QUERY], normalize_embeddings=True, convert_to_numpy=True)
sim = cosine_similarity(E, q).ravel()

boot_df = chunks_df.dropna(subset=["time_bin"]).copy().reset_index(drop=True)
boot_df["time_bin"] = boot_df["time_bin"].astype(str)
boot_df["query_sim"] = sim[boot_df.index]

time_order = sorted(boot_df["time_bin"].unique().tolist(), key=bin_start)
rng = np.random.default_rng(RANDOM_STATE)
rows = []

for bin_label, sub in boot_df.groupby("time_bin"):
    vals = sub["query_sim"].to_numpy()
    n = len(vals)

    observed = vals.mean()
    boots = np.empty(N_BOOT, dtype=float)
    for b in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        boots[b] = vals[idx].mean()

    rows.append({
        "time_bin": str(bin_label),
        "mean_similarity": observed,
        "ci_low": np.percentile(boots, 2.5),
        "ci_high": np.percentile(boots, 97.5),
        "n_chunks": n,
    })

boot_semantic_ci = pd.DataFrame(rows).sort_values("time_bin", key=lambda s: s.map(bin_start))
display(boot_semantic_ci)

plt.figure(figsize=(10, 5))
plt.plot(boot_semantic_ci["time_bin"], boot_semantic_ci["mean_similarity"], marker="o")
plt.fill_between(
    boot_semantic_ci["time_bin"],
    boot_semantic_ci["ci_low"],
    boot_semantic_ci["ci_high"],
    alpha=0.25,
)
plt.title(f'Bootstrap CI for embedding similarity to "{TARGET_QUERY}" by time bin')
plt.xlabel("Time bin")
plt.ylabel("Mean cosine similarity")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Reporting standards and alternative explanations

Before reporting a semantic-shift claim, ask:

1. **Is the pattern visible in representative chunks?**
2. **Does it survive a change in neighborhood size?**
3. **Does it survive a small change in query wording?**
4. **Could the pattern be explained by corpus composition?**
   - different authors
   - different genres
   - translated vs non-translated works
   - uneven bin sizes
5. **Are we really observing “semantic change,” or simply a shift in surrounding topics?**

A responsible report should distinguish:
- **result**: “the neighborhood around `virtue` changes after bin X”
- **interpretation**: “this may reflect a change in ethical discourse”
- **uncertainty**: “alternative explanations remain plausible”

## Save outputs

We save the key tables used in this notebook so that later work does not need to recompute them.

In [ ]:
shift_out = TABLES_DIR / 'nb12-semantic_shift_scores.csv'
overlap_out = TABLES_DIR / 'nb12-neighborhood_overlap.csv'
neighbors_out = TABLES_DIR / 'nb12-concept_neighbors.parquet'
robust_out = TABLES_DIR / 'nb12-semantic_shift_robustness.csv'
variant_out = TABLES_DIR / 'nb12-semantic_shift_query_variants.csv'

shift.to_csv(shift_out, index=False)
overlap.to_csv(overlap_out, index=False)
neighbors.to_parquet(neighbors_out, index=False)
robust_shift.to_csv(robust_out, index=False)

if len(variant_shift):
    variant_shift.to_csv(variant_out, index=False)

print('Saved:')
print('-', shift_out)
print('-', overlap_out)
print('-', neighbors_out)
print('-', robust_out)
if len(variant_shift):
    print('-', variant_out)

## Final reflection

Choose one target concept and write a short methodological paragraph answering:

- What changed, according to the neighborhood metrics?
- Which chunks best illustrate that change?
- Which alternative explanations remain plausible?
- Which robustness check increased or reduced your confidence?

This reflection matters more than the raw metric. In semantic-shift analysis, **interpretive discipline** is part of the method.

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A12 highlight;
```